In [1]:
import sys 
import pandas as pd 
import numpy as np 
import calendar
from pathlib import Path 

root = Path.cwd().parent
sys.path.insert(0, str(root))

from src.utils import df_loader, date_formatter, day_name_getter
raw_df = df_loader()

### 01. Первичный осмотр данных 

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_info_rows', 99999)

display(raw_df.head())
display(raw_df.shape)
raw_df.info()

cols_to_check_mx_mn = list(raw_df.columns) 
other_cols = ['participant_id', 'day_of_week', 'sex', 'chronotype', 'deadline_day', 'exam_period', 'day_type']

for c in other_cols:
    cols_to_check_mx_mn.remove(c)
    
for c in cols_to_check_mx_mn: 
    print(f'{c}:')
    print(f'maximun: {raw_df[c].max()} | minimun: {raw_df[c].min()}')
    print()
    
other_cols.remove('participant_id')
for c in other_cols: 
    print(f'=== {c} ===')
    print(f'unique values: {list(raw_df[c].unique())}')
    print()

,participant_id,date,day_of_week,age,sex,height_cm,weight_kg,bmi,chronotype,sleep_hours,sleep_quality_1_10,sleep_latency_min,night_awakenings,steps,exercise_minutes,workout_intensity_1_10,soreness_1_10,study_hours,work_hours,screen_hours,caffeine_mg,social_hours,stress_1_10,mood_1_10,energy_1_10,focus_1_10,productivity_score_0_100,mental_strain_0_10,resting_heart_rate,hydration_liters,calories_kcal,deadline_day,exam_period,day_type
0,P0001,2025-01-01,Wednesday,30,prefer_not_to_say,171.5,68.4,23.3,intermediate,8.73,7.0,29.6,0.0,NaN,4.0,4.2,0.0,3.00,2.99,1.92,210.0,0.59,3.3,8.6,9.9,NaN,82.7,0.7,66.0,NaN,2124.0,no,no,weekday
1,P0001,2025-01-02,Thursday,30,prefer_not_to_say,171.5,67.7,23.0,intermediate,7.48,6.5,12.2,1.0,3110.0,6.0,4.9,0.3,2.16,NaN,4.85,97.0,0.19,2.0,9.1,7.7,8.2,72.9,2.5,NaN,2.07,1331.0,no,no,weekday
2,P0001,2025-01-03,Friday,30,prefer_not_to_say,171.5,67.4,22.9,intermediate,9.15,5.2,15.4,0.0,9805.0,64.0,5.7,3.5,0.00,2.94,1.91,NaN,1.92,1.8,6.5,6.8,6.4,64.5,2.3,NaN,1.86,3074.0,no,no,weekday
3,P0001,2025-01-04,Saturday,30,prefer_not_to_say,171.5,69.2,23.5,intermediate,8.89,7.2,26.2,0.0,NaN,6.0,5.1,2.6,0.45,0.00,3.46,0.0,2.76,1.3,10.0,9.9,8.5,70.2,0.8,61.0,NaN,3427.0,no,no,weekend
4,P0001,2025-01-05,Sunday,30,prefer_not_to_say,171.5,69.8,23.7,intermediate,8.58,7.4,33.9,1.0,NaN,83.0,7.5,3.8,1.65,1.78,5.36,106.0,2.39,2.8,8.4,9.2,9.7,89.9,1.2,73.0,2.39,3016.0,no,no,weekend


(60120, 34)

<class 'pandas.DataFrame'>
RangeIndex: 60120 entries, 0 to 60119
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   participant_id            60120 non-null  str    
 1   date                      60120 non-null  str    
 2   day_of_week               60120 non-null  str    
 3   age                       60120 non-null  int64  
 4   sex                       60058 non-null  str    
 5   height_cm                 60120 non-null  float64
 6   weight_kg                 60120 non-null  float64
 7   bmi                       60120 non-null  float64
 8   chronotype                60120 non-null  str    
 9   sleep_hours               60120 non-null  float64
 10  sleep_quality_1_10        58036 non-null  float64
 11  sleep_latency_min         55960 non-null  float64
 12  night_awakenings          57356 non-null  float64
 13  steps                     56431 non-null  float64
 14  exercise_minutes 

В ходе первичного осмотра были обнаружены ***следующие проблемы:*** 
- столбец date имеет тип str 
- отрицательные значения в столбцах: sleep_hours, steps, exercise_minutes, study_hours, caffeine_mg, productivity_score_0_100, hydration_liters
- странные значения в столбцах: hydration_liters, resting_heart_rate
- значения, превыщающие допустимый максимум (100) в столбце productivity_score_0_100
- некорректные значения и значения, имеющие отличия в написании (капслок, лишние пробелы и т.д.) в столбцах: day_of_week, sex, chronotype, deadline_day, day_type

### 02. Исправление некорректных значений 

In [3]:
raw_df['date'] = raw_df['date'].apply(date_formatter)
raw_df['date'] = pd.to_datetime(raw_df['date'], format='%Y-%m-%d', errors='coerce')
round(raw_df['date'].isna().sum() / raw_df['date'].count() * 100, 3)  # 0.12% от дат имеют некорректные значения года/месяца/дня. процент слишком мал, чтобы существенно помешать дальнейшему анализу. 

np.float64(0.117)

In [4]:
neg_values_cols = ['sleep_hours', 'steps', 'exercise_minutes', 'study_hours', 'caffeine_mg', 'productivity_score_0_100', 'hydration_liters']
for c in neg_values_cols: 
    raw_df[c] = np.where(raw_df[c] < 0, 0, raw_df[c])
    
raw_df.hydration_liters = np.where(raw_df.hydration_liters > 8, np.nan, raw_df.hydration_liters)
raw_df.resting_heart_rate = np.where((raw_df.resting_heart_rate < 30) | (raw_df.resting_heart_rate > 150), np.nan, raw_df.resting_heart_rate)

raw_df.productivity_score_0_100 = np.where(raw_df.productivity_score_0_100 > 100, 100, raw_df.productivity_score_0_100)

raw_df.day_of_week = raw_df['date'].apply(day_name_getter)

raw_df.sex = raw_df.sex.str.lower().str.strip()
raw_df.sex = np.select(
    condlist=[raw_df.sex == 'nonbinary', raw_df.sex == 'prefer_not_to_say'], 
    choicelist=['non-binary', np.nan], 
    default=raw_df.sex
)

raw_df.chronotype = raw_df.chronotype.str.lower().str.strip()
raw_df.chronotype = np.where(raw_df.chronotype == 'unknown', np.nan, raw_df.chronotype)

raw_df.deadline_day = raw_df.deadline_day.str.lower().str.strip()
raw_df.deadline_day = np.where(raw_df.deadline_day == 'unknown', np.nan, raw_df.deadline_day)

raw_df.day_type = raw_df.day_type.str.lower().str.strip()
raw_df.day_type = np.where(raw_df.day_type == 'week end', 'weekend', raw_df.day_type)

### 03. Замена текстовых на True/False (где применимо)

In [5]:
d = {'yes': True, 'no': False}

raw_df.deadline_day = raw_df.deadline_day.map(d)
raw_df.exam_period = raw_df.exam_period.map(d)

raw_df.day_type = raw_df.day_type.map({'weekend': True, 'weekday': False})
raw_df = raw_df.rename(columns={'day_type': 'is_weekend'})

### 04. Повторная проверка данных 

In [6]:
for c in cols_to_check_mx_mn: 
    print(f'{c}:')
    print(f'maximun: {raw_df[c].max()} | minimun: {raw_df[c].min()}')
    print()
    
other_cols[other_cols.index('day_type')] = 'is_weekend'
for c in other_cols: 
    print(f'=== {c} ===')
    print(f'unique values: {list(raw_df[c].unique())}')
    print()

date:
maximun: 2025-12-01 00:00:00 | minimun: 2025-01-01 00:00:00

age:
maximun: 35 | minimun: 18

height_cm:
maximun: 199.0 | minimun: 150.7

weight_kg:
maximun: 230.0 | minimun: 18.0

bmi:
maximun: 61.9 | minimun: 10.1

sleep_hours:
maximun: 22.0 | minimun: 0.0

sleep_quality_1_10:
maximun: 10.0 | minimun: 1.0

sleep_latency_min:
maximun: 240.0 | minimun: 1.0

night_awakenings:
maximun: 10.0 | minimun: 0.0

steps:
maximun: 95000.0 | minimun: 0.0

exercise_minutes:
maximun: 420.0 | minimun: 0.0

workout_intensity_1_10:
maximun: 10.0 | minimun: 1.0

soreness_1_10:
maximun: 10.0 | minimun: 0.0

study_hours:
maximun: 19.0 | minimun: 0.0

work_hours:
maximun: 16.0 | minimun: 0.0

screen_hours:
maximun: 16.82 | minimun: 0.0

caffeine_mg:
maximun: 1400.0 | minimun: 0.0

social_hours:
maximun: 8.0 | minimun: 0.0

stress_1_10:
maximun: 10.0 | minimun: 1.3

mood_1_10:
maximun: 10.0 | minimun: 0.0

energy_1_10:
maximun: 10.0 | minimun: 0.0

focus_1_10:
maximun: 10.0 | minimun: 0.0

productivity

#### 05. Загрузка очищенных данных в csv 

In [7]:
raw_df.to_csv('../data/processed.csv')